# 00 Integration: validators and the Step 0 count check

Run after every fetch. Every check raises naming the column.

In [1]:
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)      # config.py uses relative paths (data/cache); nbconvert starts in notebooks/

In [2]:
import pandas as pd, config
from src.validators import validate_neurons, validate_edges, assert_interchangeable
rows = {"fafb": ("783", f"ann{config.FAFB_ANNOTATIONS_TAG[1:]}", config.FAFB_SYNAPSE_TABLE, {}),
        "malecns": ("v1.0", "traced", "traced", {"conf": config.MALECNS_SYN_CONFIDENCE})}
T = {}
for d, (rel, nsrc, esrc, kw) in rows.items():
    n = pd.read_parquet(config.cache_path(d, "neurons", rel, nsrc))
    e = pd.read_parquet(config.cache_path(d, "edges", rel, esrc, min_syn=1, **kw))
    r = pd.read_parquet(config.cache_path(d, "edgeroi", rel, esrc, min_syn=1, **kw))
    validate_neurons(n); validate_edges(e, n); validate_edges(r, n, roi=True)
    T[d] = (n, e); print(d, "validators passed:", len(n), "neurons,", len(e), "edges")
assert_interchangeable(T["fafb"][0], T["malecns"][0]); assert_interchangeable(T["fafb"][1], T["malecns"][1])
print("assert_interchangeable passed")

fafb validators passed: 139255 neurons, 15091983 edges


malecns validators passed: 165122 neurons, 25563096 edges
assert_interchangeable passed


In [3]:
expected = {"fafb neurons": 139_248, "fafb edges >= 5": 2_700_513, "malecns neurons": 165_122}
got = {"fafb neurons": len(T["fafb"][0]), "fafb edges >= 5": int((T["fafb"][1].syn_count >= 5).sum()),
       "malecns neurons": len(T["malecns"][0]), "malecns edges >= 5 (agree one number with Projects 1 and 4)": int((T["malecns"][1].syn_count >= 5).sum())}
pd.DataFrame({"got": got, "expected": {**expected, "malecns edges >= 5 (agree one number with Projects 1 and 4)": None}})

,got,expected
fafb neurons,139255,139248.0
fafb edges >= 5,2700513,2700513.0
malecns neurons,165122,165122.0
malecns edges >= 5 (agree one number with Projects 1 and 4),6235649,NaN
